# Dataset Profiling

This notebook captures the first exploratory look at the raw hospital patient flow dataset. It is intentionally lightweight: no cleaning, renaming, deduplication, or business logic is applied here.

Structured pipeline work belongs in `src/`. Bronze is responsible for raw inventory and metadata generation, while Silver handles schema and type normalization later in the project flow.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

## Project Path Setup

Add the project `src/` directory to the notebook path so discovery helpers can be reused instead of copying pipeline logic into the notebook.

In [ ]:
PROJECT_ROOT = Path.cwd()
project_candidate = PROJECT_ROOT / "projects" / "01-hospital-analytics"
if project_candidate.exists():
    PROJECT_ROOT = project_candidate
else:
    while PROJECT_ROOT.name != "01-hospital-analytics" and PROJECT_ROOT != PROJECT_ROOT.parent:
        PROJECT_ROOT = PROJECT_ROOT.parent

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

PROJECT_ROOT

In [ ]:
from ingestion.raw_inventory import (
    get_raw_data_dir,
    list_raw_files,
    list_supported_data_files,
    path_relative_to_project,
)
from processing.bronze.profiling import select_main_csv_file

## Raw Dataset Discovery

List the raw landing area exactly as Bronze sees it, including operational marker files. The main analytical CSV is selected with the same deterministic helper used by Bronze profiling.

In [ ]:
raw_data_dir = get_raw_data_dir()
raw_files = list_raw_files(raw_data_dir)
supported_data_files = list_supported_data_files(raw_data_dir)
main_csv_path = select_main_csv_file(supported_data_files)

raw_data_dir, main_csv_path

In [ ]:
raw_file_inventory = pd.DataFrame(
    [
        {
            "path": path_relative_to_project(path),
            "size_bytes": path.stat().st_size,
            "is_supported_data_file": path in supported_data_files,
        }
        for path in raw_files
    ]
)

raw_file_inventory

## Read the Raw CSV

Read the selected raw CSV with Pandas for exploration only. The raw column names and values are left unchanged.

In [ ]:
raw_df = pd.read_csv(main_csv_path)
raw_df.head()

In [ ]:
raw_df.shape

In [ ]:
list(raw_df.columns)

In [ ]:
raw_df.info()

## Basic Profiling Signals

These checks describe the raw dataset. They do not decide how records should be cleaned or modeled.

In [ ]:
raw_df.isna().sum().sort_values(ascending=False)

In [ ]:
duplicate_row_count = int(raw_df.duplicated().sum())
duplicate_row_count

In [ ]:
cardinality_review = (
    raw_df.nunique(dropna=False)
    .sort_values(ascending=False)
    .rename("distinct_values_including_null")
    .to_frame()
)

cardinality_review

## Notes

- This notebook is exploratory profiling only.
- No cleaning is performed here.
- Bronze captures raw inventory and profiling metadata in a repeatable job.
- Silver owns structured cleaning, type normalization, and row-level conformance.
- Gold owns analytical aggregations for future serving and dashboard use.